# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, duckdb, os
BASE = "/content/drive/MyDrive/flyrank-internship/work/outputs"

feature_vector = pd.read_csv(f"{BASE}/feature_vector.csv")
labels = pd.read_csv(f"{BASE}/labels.csv")
ctr_check = pd.read_csv(f"{BASE}/ctr_check.csv")
# only load what THIS notebook actually needs — no need to load everything every time

Mounted at /content/drive


In [2]:
from huggingface_hub import login

login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [6]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [5]:
from datasets import load_dataset
import duckdb


fact_content_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train")
fact_content = fact_content_ds.data.table   # Arrow, not pandas — much lighter

dim_content_ds = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train")
dim_content = dim_content_ds.data.table

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [7]:

import pandas as pd

key_fields = ["word_count", "search_volume", "content_age_days", "gsc_avg_position_prior"]
dist_summary = feature_vector[key_fields].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99])
dist_summary

,word_count,search_volume,content_age_days,gsc_avg_position_prior
count,303332.000000,253687.000000,303332.000000,303332.000000
mean,2323.585319,154.444059,197.181076,10.237045
std,1018.717166,2187.333442,109.102212,10.082779
min,0.000000,0.000000,0.000000,0.000000
1%,730.000000,0.000000,5.000000,0.872471
25%,1594.000000,0.000000,128.000000,7.877132
50%,2330.000000,10.000000,204.000000,7.877132
75%,2674.000000,20.000000,271.000000,7.877132
95%,4127.000000,320.000000,382.000000,28.954219
99%,5985.000000,2400.000000,448.000000,57.822793


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [8]:
# --- Signal 1: Staleness (already tested in Week 4 — reused here formally) ---
age_check = duckdb.sql("""
    SELECT
        CASE WHEN content_age_days < 90 THEN '<90d'
             WHEN content_age_days < 180 THEN '90-180d'
             ELSE '180d+' END AS age_bucket,
        COUNT(*) AS n, AVG(declined) AS pct_declined
    FROM (SELECT f.content_age_days, l.declined
          FROM feature_vector f JOIN labels l USING (client_hash_id, content_hash_id))
    GROUP BY age_bucket ORDER BY age_bucket
""").df()
print(age_check)

  age_bucket      n  pct_declined
0      180d+  68735      0.205543
1    90-180d  25785      0.192127
2       <90d  39566      0.134686


In [9]:
# --- Signal 2: CTR-underperformance, volume-floored (corrected version from Week 4) ---
merged_ctr = ctr_check.merge(labels, on=["client_hash_id", "content_hash_id"])
has_volume = merged_ctr["actual_ctr"] > merged_ctr["actual_ctr"].median()
ctr_bucket_v2 = merged_ctr[has_volume].groupby("underperforming")["declined"].agg(["mean", "count"])
print(ctr_bucket_v2)

                     mean  count
underperforming                 
False            0.558650  18670
True             0.404283  34696


In [10]:
# --- Signal 3: NEW — search_volume vs. decline ---
volume_check = duckdb.sql("""
    SELECT
        CASE WHEN search_volume < 10 THEN 'low'
             WHEN search_volume < 500 THEN 'medium'
             ELSE 'high' END AS volume_bucket,
        COUNT(*) AS n, AVG(declined) AS pct_declined
    FROM (SELECT f.search_volume, l.declined
          FROM feature_vector f JOIN labels l USING (client_hash_id, content_hash_id))
    GROUP BY volume_bucket ORDER BY volume_bucket
""").df()
print(volume_check)

  volume_bucket      n  pct_declined
0          high  15388      0.129906
1           low  46338      0.178255
2        medium  72360      0.195578


**`What this shows:`** it's not a clean, one-direction trend like staleness was. Decline rate actually goes up from low to medium, then drops back down for high-volume pages. That's a bump shape, not a straight line — medium-volume pages are the most likely to decline, while high-volume pages (the most valuable keywords) decline the least.

**Verdict: MIXED.**

 Not confirmed (no clean, single-direction pattern like staleness), not opposite (it's not simply backwards either), and not false (there clearly is some real difference between buckets — 13% vs 19.6% isn't nothing). It's a real but non-linear relationship, which doesn't fit neatly into "yes this works" or "no it doesn't."





## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [11]:
# Staleness is the signal behind FlyRank's real "refresh" flag.
# Test the flag's actual assumption: does a real staleness threshold (180+ days)
# predict decline meaningfully better than a naive 50/50 split would?
flag_test = duckdb.sql("""
    SELECT
        content_age_days >= 180 AS flagged_stale,
        COUNT(*) AS n, AVG(declined) AS pct_declined
    FROM (SELECT f.content_age_days, l.declined
          FROM feature_vector f JOIN labels l USING (client_hash_id, content_hash_id))
    GROUP BY flagged_stale
""").df()
print(flag_test)

   flagged_stale      n  pct_declined
0          False  65351      0.157350
1           True  68735      0.205543


**`What this shows:`**

 pages that FlyRank's real refresh flag would mark as "stale" (180+ days old) decline at a meaningfully higher rate — 20.6% versus 15.7% for newer pages. That's roughly a 5-point gap, and it's in the direction you'd expect, consistent with what you already found in the finer-grained age buckets earlier.

## **Verdict for the flag-linked test:**

 `CONFIRMED`. The actual threshold FlyRank's production flag uses (180 days) does capture a real, meaningful difference in decline behavior — it's not an arbitrary cutoff, the data backs it up.

Flag tested: FlyRank's refresh flag treats pages 180+ days old as stale and worth reviewing. Testing this exact threshold against real decline rates confirms it: flagged pages decline at 20.6% versus 15.7% for unflagged pages — a real, meaningful gap. This means the production flag's underlying assumption is sound and worth trusting as a starting point, not something that needs to be second-guessed or replaced.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team can trust FlyRank's existing age-based stale flag — the data confirms older pages really do decline more, both in fine-grained buckets and at the exact 180-day threshold the flag uses. Raw CTR-underperformance and search volume are less reliable as standalone signals: CTR only works once you filter for pages with real traffic (otherwise low-traffic pages falsely look "safe" due to a floor effect), and search volume shows a non-linear pattern that doesn't translate cleanly into a simple rule. In practice: lead with age as the primary trigger, and treat CTR and volume as secondary context rather than standalone flags

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.